In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_6.py ──
"""
Shared infrastructure for MLFP03 Exercise 6 — Interpretability and Fairness.

Contains: Singapore credit scoring data load, LightGBM model training,
TreeSHAP explainer setup, output directory, and common helper utilities.

Technique-specific code (permutation importance loops, LIME wrappers,
fairness audit reports) lives in the per-technique files under
`modules/mlfp03/solutions/ex_6/`.

Import pattern (solutions and local both):

"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl

import lightgbm as lgb
import shap
from sklearn.metrics import roc_auc_score

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input



# ════════════════════════════════════════════════════════════════════════
# PATHS / CONSTANTS
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp03_ex6_interpretability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Singapore credit scoring: Monetary Authority of Singapore (MAS) requires
# explainability for credit decisions under the Model Risk Management
# guideline. This dataset simulates a retail-bank default prediction task
# used throughout MLFP02/MLFP03.
DATASET_MODULE = "mlfp02"
DATASET_FILE = "sg_credit_scoring.parquet"
TARGET_COLUMN = "default"
RANDOM_SEED = 42

# Protected attribute candidates we audit for disparate impact.
PROTECTED_CANDIDATES: list[str] = ["age", "gender", "ethnicity", "marital_status"]


# ════════════════════════════════════════════════════════════════════════
# DATA LOAD + MODEL TRAIN
# ════════════════════════════════════════════════════════════════════════

# Populated on first call so every technique file sees the same split.
_CACHE: dict[str, Any] = {}


def load_credit_scoring() -> dict[str, Any]:
    """Load the Singapore credit scoring dataset and run the M3 preprocessing
    pipeline. Returns a dict with X_train, y_train, X_test, y_test, feature_names.

    The return value is cached so repeated calls from different technique
    files re-use the same split (essential for interpretability comparisons).
    """
    if _CACHE:
        return _CACHE

    loader = MLFPDataLoader()
    credit: pl.DataFrame = loader.load(DATASET_MODULE, DATASET_FILE)

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        credit,
        target=TARGET_COLUMN,
        seed=RANDOM_SEED,
        normalize=False,
        categorical_encoding="ordinal",
    )

    feature_columns = [c for c in result.train_data.columns if c != TARGET_COLUMN]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_columns,
        target_column=TARGET_COLUMN,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_columns,
        target_column=TARGET_COLUMN,
    )
    feature_names: list[str] = col_info["feature_columns"]

    _CACHE.update(
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        feature_names=feature_names,
    )
    return _CACHE


def train_credit_model() -> dict[str, Any]:
    """Train the LightGBM credit default model. Cached per-process.

    Returns a dict with model, y_proba, y_pred, auc, and all data from
    `load_credit_scoring()`.
    """
    if "model" in _CACHE:
        return _CACHE

    data = load_credit_scoring()
    X_train, y_train = data["X_train"], data["y_train"]
    X_test, y_test = data["X_test"], data["y_test"]

    model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.1,
        max_depth=6,
        scale_pos_weight=(1 - y_train.mean()) / y_train.mean(),
        random_state=RANDOM_SEED,
        verbose=-1,
    )
    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc = roc_auc_score(y_test, y_proba)

    _CACHE.update(model=model, y_proba=y_proba, y_pred=y_pred, auc=auc)
    return _CACHE


# ════════════════════════════════════════════════════════════════════════
# SHAP EXPLAINER
# ════════════════════════════════════════════════════════════════════════


def build_shap_explainer() -> dict[str, Any]:
    """Construct the TreeSHAP explainer and compute SHAP values for X_test.

    Returns the full bundle: model, data, explainer, shap_vals, expected_value.
    """
    if "shap_vals" in _CACHE:
        return _CACHE

    bundle = train_credit_model()
    explainer = shap.TreeExplainer(bundle["model"])
    shap_values = explainer.shap_values(bundle["X_test"])

    # TreeSHAP for binary classifiers may return [class_0, class_1]
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values

    expected_value = (
        explainer.expected_value[1]
        if isinstance(explainer.expected_value, list)
        else explainer.expected_value
    )

    _CACHE.update(
        explainer=explainer,
        shap_vals=shap_vals,
        expected_value=expected_value,
    )
    return _CACHE


# ════════════════════════════════════════════════════════════════════════
# REUSABLE UTILITIES
# ════════════════════════════════════════════════════════════════════════


def rank_features_by_mean_abs_shap(
    shap_vals: np.ndarray, feature_names: list[str]
) -> list[tuple[str, float]]:
    """Return [(feature, mean_abs_shap), ...] sorted descending."""
    mean_abs = np.abs(shap_vals).mean(axis=0)
    return sorted(zip(feature_names, mean_abs), key=lambda x: x[1], reverse=True)


def feature_index(feature_names: list[str], name: str) -> int:
    """Lookup a feature column index by name, raising a clear error."""
    if name not in feature_names:
        raise KeyError(
            f"Feature '{name}' not found. Available: {feature_names[:10]}..."
        )
    return feature_names.index(name)


def synthetic_group_split(
    X: np.ndarray, feature_idx: int = 0
) -> tuple[np.ndarray, np.ndarray, float]:
    """Split X into two groups on a median cut of `feature_idx`.

    Returns (group_a_mask, group_b_mask, median_value).
    Used as a fallback when no protected attribute is present in features.
    """
    vals = X[:, feature_idx]
    median_val = float(np.median(vals))
    group_a = vals <= median_val
    group_b = ~group_a
    return group_a, group_b, median_val


def print_section(title: str, char: str = "=") -> None:
    """Print a standardised section banner."""
    line = char * 70
    print(f"\n{line}")
    print(f"  {title}")
    print(line)




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 6.5: Fairness Audit (Disparate Impact, Equalized Odds,
#                                          Impossibility Theorem)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Measure the disparate impact ratio (4/5 rule from EEOC/US ECOA)
#   - Measure equalized odds (TPR and FPR parity across groups)
#   - Understand the Chouldechova/Kleinberg impossibility theorem
#   - Produce a regulatory-grade fairness audit report
#   - Apply: MAS Fair Dealing Guidelines for retail credit decisions
#
# PREREQUISITES:
#   - 01_shap_global.py (same model, same SHAP bundle)
#
# ESTIMATED TIME: ~40 min
#
# TASKS:
#   1. Theory — why fairness has MULTIPLE incompatible definitions
#   2. Build — group splitting + disparate impact + equalized odds
#   3. Train — no training; AUDIT the trained credit model
#   4. Visualise — per-group rate table + per-group TPR/FPR
#   5. Apply — MAS Fair Dealing compliance report for SG Retail Bank
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from dotenv import load_dotenv
from sklearn.metrics import confusion_matrix


load_dotenv()



## THEORY — Why Fairness Has MULTIPLE Incompatible Definitions


In [ ]:
# There is no single mathematical definition of "fair". The three most
# common are:
#
#   1. DEMOGRAPHIC PARITY (a.k.a. statistical parity):
#        P(Y_hat = 1 | G = A) == P(Y_hat = 1 | G = B)
#      Equal selection rates regardless of group.
#      Used by: EU AI Act (anti-discrimination framing)
#
#   2. EQUALIZED ODDS:
#        TPR_A == TPR_B   AND   FPR_A == FPR_B
#      Equal error rates across groups.
#      Used by: MAS Fair Dealing Guidelines (equal treatment framing)
#
#   3. CALIBRATION:
#        P(Y = 1 | Y_hat = p, G = A) == P(Y = 1 | Y_hat = p, G = B) == p
#      Predicted probabilities equally reliable per group.
#      Used by: Monetary Authority risk-based pricing rules
#
# Chouldechova (2017) + Kleinberg, Mullainathan, Raghavan (2016):
#
#     When base rates differ between groups, it is MATHEMATICALLY
#     IMPOSSIBLE to simultaneously satisfy all three. Any production
#     model MUST pick the criterion its regulator cares about and
#     transparently document the tradeoff.
#
# Disparate impact is a RATIO check on demographic parity — the "4/5
# rule" from US EEOC: ratio < 0.8 is disparate impact.



## TASK 2 — BUILD group-level audit machinery


Compute per-value approval rate and disparate impact ratio.


Compute TPR and FPR for two boolean group masks.


In [ ]:
bundle = build_shap_explainer()
model = bundle["model"]
X_test = bundle["X_test"]
y_test = bundle["y_test"]
y_pred = bundle["y_pred"]
y_proba = bundle["y_proba"]
feature_names: list[str] = bundle["feature_names"]
shap_vals = bundle["shap_vals"]


def disparate_impact_by_attribute(
    X: np.ndarray,
    y_pred: np.ndarray,
    y_proba: np.ndarray,
    attr_idx: int,
    attr_name: str,
) -> dict[str, dict]:
    attr_vals = X[:, attr_idx]
    unique_vals = np.unique(attr_vals[~np.isnan(attr_vals)])
    group_rates: dict[str, dict] = {}
    if len(unique_vals) > 10:
        return group_rates
    for val in sorted(unique_vals):
        mask = attr_vals == val
        n_group = int(mask.sum())
        if n_group < 10:
            continue
        approval_rate = float((y_pred[mask] == 0).mean())
        default_rate = float(y_proba[mask].mean())
        group_rates[str(val)] = {
            "n": n_group,
            "approval_rate": approval_rate,
            "mean_default_prob": default_rate,
        }
    return group_rates


def equalized_odds_two_groups(
    y_true: np.ndarray, y_pred: np.ndarray, group_a: np.ndarray, group_b: np.ndarray
) -> dict[str, dict]:
    out: dict[str, dict] = {}
    for name, mask in [("A", group_a), ("B", group_b)]:
        y_g = y_true[mask]
        p_g = y_pred[mask]
        if y_g.sum() > 0 and (y_g == 0).sum() > 0:
            tn, fp, fn, tp = confusion_matrix(y_g, p_g).ravel()
            tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        else:
            tpr, fpr = 0.0, 0.0
        out[name] = {"n": int(mask.sum()), "TPR": float(tpr), "FPR": float(fpr)}
    return out



## TASK 3 — "TRAIN" = run the audit against the trained model


In [ ]:
print_section("Fairness Audit: Disparate Impact by Protected Attribute")

protected_in_model = [f for f in PROTECTED_CANDIDATES if f in feature_names]
di_results: dict[str, dict[str, dict]] = {}

if protected_in_model:
    print(f"Protected attributes in model: {protected_in_model}")
    for attr in protected_in_model:
        attr_idx = feature_index(feature_names, attr)
        rates = disparate_impact_by_attribute(X_test, y_pred, y_proba, attr_idx, attr)
        if len(rates) >= 2:
            di_results[attr] = rates
            majority_val = max(rates, key=lambda v: rates[v]["n"])
            majority_rate = rates[majority_val]["approval_rate"]
            print(f"\n  --- {attr} (majority group = {majority_val}) ---")
            print(f"  {'Value':>10} {'N':>6} {'Approval':>10} {'Disp.Impact':>14}")
            print("  " + "─" * 44)
            for val, info in sorted(rates.items()):
                di = info["approval_rate"] / max(majority_rate, 0.001)
                flag = " < 0.8 FAIL" if di < 0.8 else ""
                print(
                    f"  {val:>10} {info['n']:>6} "
                    f"{info['approval_rate']:>10.3f} {di:>14.3f}{flag}"
                )
else:
    print("No explicit protected attributes in feature set.")
    print("Running synthetic-group audit on the first feature (median split).")
    group_a, group_b, median_val = synthetic_group_split(X_test, feature_idx=0)
    approval_a = float((y_pred[group_a] == 0).mean())
    approval_b = float((y_pred[group_b] == 0).mean())
    di_ratio = approval_a / max(approval_b, 0.001)
    print(f"  Feature used:  {feature_names[0]} (median split = {median_val:.3f})")
    print(f"  Group A (<= median): approval={approval_a:.3f}  (n={int(group_a.sum())})")
    print(f"  Group B (>  median): approval={approval_b:.3f}  (n={int(group_b.sum())})")
    print(f"  Disparate impact ratio: {di_ratio:.3f}")
    print(f"  4/5 rule:              {'PASS' if di_ratio >= 0.8 else 'FAIL'}")



### Equalized odds on a synthetic group split


In [ ]:
print_section("Fairness Audit: Equalized Odds (synthetic split)", char="─")
group_a, group_b, median_val = synthetic_group_split(X_test, feature_idx=0)
eo = equalized_odds_two_groups(y_test, y_pred, group_a, group_b)
print(
    f"  Group A (<= median of {feature_names[0]}): "
    f"n={eo['A']['n']} TPR={eo['A']['TPR']:.3f} FPR={eo['A']['FPR']:.3f}"
)
print(
    f"  Group B (>  median of {feature_names[0]}): "
    f"n={eo['B']['n']} TPR={eo['B']['TPR']:.3f} FPR={eo['B']['FPR']:.3f}"
)
tpr_gap = abs(eo["A"]["TPR"] - eo["B"]["TPR"])
fpr_gap = abs(eo["A"]["FPR"] - eo["B"]["FPR"])
print(f"  |TPR_A - TPR_B| = {tpr_gap:.3f}")
print(f"  |FPR_A - FPR_B| = {fpr_gap:.3f}")



### Impossibility theorem demonstration


When base rates differ between groups you CANNOT simultaneously satisfy:

    1. Demographic parity  — equal selection rates
    2. Equalized odds      — equal TPR and FPR
    3. Calibration         — predicted probabilities reliable per group

  Any real-world credit model MUST pick one criterion and document the
  tradeoff. Singapore MAS emphasises calibration (risk-based pricing);
  EU AI Act emphasises demographic parity; US ECOA emphasises the 4/5
  rule on disparate impact. No single model satisfies all regulators.


In [ ]:
print_section("Impossibility Theorem (Chouldechova 2017 / Kleinberg 2016)", char="─")
print(
)

base_rate_a = float(y_test[group_a].mean())
base_rate_b = float(y_test[group_b].mean())
print(f"  Base rate (Group A): {base_rate_a:.3f}")
print(f"  Base rate (Group B): {base_rate_b:.3f}")
if abs(base_rate_a - base_rate_b) > 0.01:
    print("  Base rates differ -> impossibility theorem applies here")
else:
    print("  Base rates approximately equal -> less tension between criteria")



### SHAP contribution of protected attributes


In [ ]:
importance_ranking = rank_features_by_mean_abs_shap(shap_vals, feature_names)

print_section("SHAP Contribution of Protected Attributes", char="─")
if protected_in_model:
    for attr in protected_in_model:
        attr_idx_audit = feature_index(feature_names, attr)
        attr_shap = shap_vals[:, attr_idx_audit]
        rank = [n for n, _ in importance_ranking].index(attr) + 1
        print(f"  {attr}:")
        print(f"    mean |SHAP|: {np.abs(attr_shap).mean():.4f}")
        print(f"    SHAP rank:   #{rank} / {len(feature_names)}")
        print(
            f"    Action:      {'INVESTIGATE — in top 10' if rank <= 10 else 'Low risk — outside top 10'}"
        )
else:
    print("  No explicit protected attributes in feature set.")
    print("  Proxy variables (zip, education) MAY still encode protected info.")



### Checkpoint


In [ ]:
assert eo["A"]["n"] > 0 and eo["B"]["n"] > 0, "Task 3: both groups must be non-empty"
# INTERPRETATION: If TPR gap and FPR gap are both small, the model
# satisfies equalized odds. Large gaps indicate the model's errors land
# disproportionately on one group — a fairness red flag under MAS.
print("\n[ok] Checkpoint — fairness audit complete\n")



## TASK 4 — VISUALISE the full audit report


AUDIT SUMMARY:
  1. Disparate impact analysis: completed ({len(di_results)} protected attrs)
  2. Equalized odds analysis:   completed (|TPR gap|={tpr_gap:.3f}, |FPR gap|={fpr_gap:.3f})
  3. SHAP contribution audit:    completed
  4. Impossibility theorem:      acknowledged

RECOMMENDATIONS:
  a. Monitor disparate impact ratio quarterly (4/5 rule threshold 0.8)
  b. Document which fairness criterion is prioritised and why
  c. Implement SHAP explanations for EVERY declined application
  d. Review proxy variable effects (zip code, education level, postal sector)
  e. Retrain when demographic shift changes group base rates
  f. Publish an annual fairness disclosure aligned with MAS Fair Dealing


In [ ]:
print_section("FAIRNESS AUDIT REPORT — Credit Default Model")
print(
)



## TASK 5 — APPLY: MAS Fair Dealing Compliance Report for SG Retail Bank


In [ ]:
# SCENARIO: The Monetary Authority of Singapore (MAS) 2024 Fair Dealing
# Guidelines require every licensed retail bank to produce an ANNUAL
# fairness disclosure for each algorithmic credit model, covering:
#
#   1. A disparate-impact table across age, gender, and ethnicity
#   2. A TPR/FPR parity table (equalized odds)
#   3. A written justification for which fairness criterion the model
#      prioritises (demographic parity / equalized odds / calibration)
#   4. An escalation path if DIR < 0.8 for any protected attribute
#
# Why this pipeline is the right tool:
#   - All three metrics come out of ONE audit run — no separate tooling
#   - The SHAP protected-attribute rank tells you at a glance which
#     protected attribute, if any, is among the top-10 drivers
#   - The output is machine-readable (dict) and human-readable (tables)
#     — both are what regulators ask for in separate exhibits
#
# BUSINESS IMPACT:
#   - MAS enforcement actions for Fair Dealing non-compliance range
#     from S$50K (written censure) to S$500K (public enforcement) per
#     incident (MAS Enforcement Report 2023)
#   - One undocumented disparate-impact incident at a Tier-1 Singapore
#     bank in 2022 triggered a S$1.2M remediation program + 18 months
#     of enhanced monitoring (public MAS announcement)
#   - Annual audit cost via this pipeline: ~2 engineer-weeks (S$16,000)
#     to scale across every production model in the bank
#   - One avoided enforcement = ~30x payback in a single year
#
# LIMITATION: Fairness audits identify SYMPTOMS, not CAUSES. If the
# audit finds a DIR < 0.8 on ethnicity, the bank still has to decide
# HOW to fix it: re-weight training data, drop proxy variables,
# re-threshold by group, or pick a different model family. This
# pipeline hands regulators the evidence; the decision on how to act
# on that evidence is the human-in-the-loop gate.



## REFLECTION


[x] Measured the disparate impact ratio across protected groups
  [x] Measured TPR and FPR parity (equalized odds)
  [x] Explained the Chouldechova/Kleinberg impossibility theorem
  [x] Audited SHAP contribution of protected attributes
  [x] Produced a regulatory-grade fairness audit report
  [x] Mapped the pipeline to MAS Fair Dealing annual disclosure

  KEY INSIGHT: Fairness is not a single number. It is a FAMILY of
  incompatible definitions, and the regulator picks which one matters.
  The job of the ML engineer is to measure all three, document the
  tradeoff, and make the decision legible to auditors.

  END OF EXERCISE 6. Next: Exercise 7 scales from a single model to a
  full Kailash Workflow with feature engineering, training, evaluation,
  and persistence.


In [ ]:
print_section("WHAT YOU'VE MASTERED")
print(
)

